In [23]:
import json
import os
import pandas as pd
%load_ext autoreload
%autoreload 2
from dotenv import load_dotenv
from constant import *
from GeminiModel import GeminiModel
from TrainStrategy import TrainStrategy
from LlmSatdOutputLabelConverter import LlmSatdOutputLabelConverter

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
load_dotenv()

True

In [25]:
prompt_template = PromptTemplate(
    name="Manually Crafted",
    definition="You are a Code Analysis Expert specialized in detecting Self-Admitted Technical Debt (SATD) in Java test code comments. SATD refers to comments where developers acknowledge that the current test implementation is incomplete, suboptimal, or relies on a compromise that should be addressed in the future. These admissions often appear as markers such as TODO or FIXME, or as notes about unresolved issues, temporary fixes, workarounds, hacks, performance limitations, use of deprecated APIs, unsupported features, poor design choices, skipped tests, or uncertain functionality. However, comments that only describe expected behavior of test code are not SATD unless there is additional information indicating the need for future improvement.",
    instruction="Think step by step and assign the label of yes or no for each given test code comment.",
    n_shot_template='Comment: "{{ text }}"',
    n_shot_answer_template="Answer: {{ cot }} The answer is {{ label }}.",
    line_m_before=3,
    line_n_after=3)
output_label_converter = LlmSatdOutputLabelConverter({'yes', 'no'}, DEFAULT_DETECTION_CLASS)

# Submit Gemini Batch

In [37]:
for shots in [2]:
    for model_name in ['models/gemini-2.0-flash']:
        gemini_model = GeminiModel('detect', model_name, output_label_converter, True)
        gemini_model.fit(detect_n_shot_dataset)
        gemini_model.submit_batch(detect_test_dataset.select(range(2)), DATASET_NAME, prompt_template, TrainStrategy.N_SHOT_TOP, shots, verbose=False)

models/gemini-2.0-flash-2-shot
Created batch job: batches/hopwucldckeqnea1wd2e41odls4ab5hn7jy1


In [ ]:
def parse_batch_output(model:GeminiModel, file):
    df = pd.concat([detect_train_df, detect_test_df]) if model.task_type == 'detect' else pd.concat([classify_train_df, classify_test_df])
    batch_dataset = Dataset.from_pandas(df)
    id_to_index = {batch_dataset['id'][index]: index for index in range(batch_dataset.num_rows)}
    with open(file, 'r', encoding='utf-8') as json_output_file:
                            for line in json_output_file:
                                output = json.loads((line.strip()))
                                sample_id = int(output['key'])
                                raw_predicted_label = output['response']['candidates'][-1]['content']['parts'][-1]['text']
                                predicted_label = model.output_label_converter.convert_label(raw_predicted_label)
                                model_suffix = model_uri[model_uri.rfind('-',0, len(model_uri) - len('-shot')) + 1:]
                                model.append_into_merged_file(model_suffix, batch_dataset, id_to_index[sample_id], predicted_label,raw_predicted_label)

parse_batch_output(GeminiModel('detect', 'models/gemini-2.0-flash', output_label_converter, True), '/home/cs/grad/islams32/dev/project/academic/technical-debt/cache/output/batch/output/detect_gemini-2.0-flash-2-shot.jsonl')

In [ ]:
import time

completed_states = set([
    'JOB_STATE_SUCCEEDED',
    'JOB_STATE_FAILED',
    'JOB_STATE_CANCELLED',
    'JOB_STATE_EXPIRED',
])

tryAgain = True
while tryAgain:
    tryAgain = False
    job_df = pd.read_csv(gemini_model.get_batch_job_file_name(), dtype={"status": "string"})
    for idx, row in job_df.iterrows():
        model_uri = row["model_uri"]
        if 'gemini' in  model_uri and row["status"] not in completed_states:
            tryAgain = True
            gemini_model = GeminiModel(row['task_type'], model_uri, output_label_converter, True)
            batch_job = gemini_model.model.batches.get(name=row["job_id"])
            if batch_job.state.name == 'JOB_STATE_SUCCEEDED':
                    if batch_job.dest and batch_job.dest.file_name:
                        # Results are in a file
                        result_file_name = batch_job.dest.file_name
                        print(f"Results are in file: {result_file_name}")

                        print("Downloading result file content...")
                        file_content = gemini_model.model.files.download(file=result_file_name)
                        output_file_name = row['input_file'].replace('/input/', '/output/')
                        os.makedirs(os.path.dirname(output_file_name), exist_ok=True)
                        with open(output_file_name, 'w') as output_file:
                            output_file.write(file_content.decode('utf-8'))
                        parse_batch_output(gemini_model, output_file_name)
            else:
                print(f"Job status detail {batch_job}")
            job_df.at[idx, "status"] = batch_job.state.name
    job_df.to_csv(gemini_model.get_batch_job_file_name(), index=False)
    if tryAgain:
        print('trying..')
        time.sleep(30)
